<a href="https://colab.research.google.com/github/vidalcompany/manager/blob/main/%D8%B1%D8%A8%D8%A7%D8%AA_%D8%AA%D9%84%DA%AF%D8%B1%D8%A7%D9%85_V3EEED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
V3EEED - ربات تعاملی و خودکار مدیریت و مانیتورینگ
این نسخه به صورت تعاملی در اولین اجرا اطلاعات اتصال را پرسیده و ذخیره می‌کند.
"""

import time
import json
import os
import sys
import uuid
import base64
import urllib.parse
import requests

CONFIG_FILE = "vortex_config.json"

def run_interactive_setup():
    """محیط نصب تعاملی در ترمینال لینوکس برای اولین اجرای ربات V3EEED"""
    print("\n" + "="*50)
    print(" 🕵️‍♂️ V3EEED - SETUP WIZARD")
    print(" به سیستم راه‌اندازی تعاملی ربات V3EEED خوش آمدید.")
    print("="*50 + "\n")

    config = {}

    # دریافت توکن تلگرام
    while True:
        token = input("1. توکن ربات تلگرام خود را وارد کنید:\n=> ").strip()
        if token:
            config["telegram_token"] = token
            break
        print("❌ توکن نمی‌تواند خالی باشد!")

    # دریافت چت آی‌دی ادمین
    while True:
        chat_id_raw = input("\n2. چت آی‌دی شخصی خود را وارد کنید (فقط اعداد):\n=> ").strip()
        try:
            config["allowed_chat_id"] = int(chat_id_raw)
            break
        except ValueError:
            print("❌ چت آی‌دی باید به صورت عدد وارد شود!")

    # دریافت آدرس پنل سنایی
    while True:
        url = input("\n3. آدرس کامل پنل ۳X-UI را همراه با پورت وارد کنید:\n(مثال: http://127.0.0.1:2053)\n=> ").strip()
        if url:
            # حذف اسلش آخر در صورت وجود
            if url.endswith("/"):
                url = url[:-1]
            config["panel_url"] = url
            break
        print("❌ آدرس پنل نمی‌تواند خالی باشد!")

    # دریافت یوزرنیم پنل
    username = input("\n4. نام کاربری ادمین پنل سنایی را وارد کنید [پیش‌فرض: admin]:\n=> ").strip()
    config["panel_username"] = username if username else "admin"

    # دریافت پسورد پنل
    while True:
        password = input("\n5. کلمه عبور (پسورد) پنل سنایی را وارد کنید:\n=> ").strip()
        if password:
            config["panel_password"] = password
            break
        print("❌ کلمه عبور نمی‌تواند خالی باشد!")

    # ذخیره در فایل کانفیگ محلی
    try:
        with open(CONFIG_FILE, "w", encoding="utf-8") as f:
            json.dump(config, f, ensure_ascii=False, indent=4)
        print("\n" + "="*50)
        print("✅ تنظیمات با موفقیت در فایل vortex_config.json ذخیره شد!")
        print(" ربات V3EEED از هم‌اکنون آماده شروع به کار است.")
        print("="*50 + "\n")
        return config
    except Exception as e:
        print(f"❌ خطا در ذخیره‌سازی فایل تنظیمات: {e}")
        sys.exit(1)


# بررسی وجود فایل تنظیمات یا اجرای ویزارد نصب
if not os.path.exists(CONFIG_FILE):
    config_data = run_interactive_setup()
else:
    try:
        with open(CONFIG_FILE, "r", encoding="utf-8") as f:
            config_data = json.load(f)
    except Exception as e:
        print(f"❌ خطا در خواندن فایل تنظیمات محلی: {e}")
        print("جهت راه‌اندازی مجدد فایل vortex_config.json را حذف کنید.")
        sys.exit(1)

# استخراج پارامترها از فایل پیکربندی
TELEGRAM_TOKEN = config_data.get("telegram_token")
ALLOWED_CHAT_ID = config_data.get("allowed_chat_id")
PANEL_URL = config_data.get("panel_url")
PANEL_USERNAME = config_data.get("panel_username", "admin")
PANEL_PASSWORD = config_data.get("panel_password")

# وضعیت‌های موقت برای ساخت اکانت تعاملی (FSM)
USER_STATES = {}

class VortexBridge:
    def __init__(self):
        self.session = requests.Session()
        self.last_update_id = 0
        self.loaded_inbounds = []

    def login_to_panel(self):
        """ورود محلی به پنل ۳X-UI برای دریافت نشست فعال"""
        login_url = f"{PANEL_URL}/login"
        payload = {
            "username": PANEL_USERNAME,
            "password": PANEL_PASSWORD
        }
        try:
            response = self.session.post(login_url, data=payload, timeout=10)
            if response.ok and response.json().get("success"):
                return True
            return False
        except Exception as e:
            print(f"Error logging in locally: {e}")
            return False

    def fetch_inbounds_data(self):
        """دریافت زنده اینباندها و کاربران"""
        if not self.login_to_panel():
            return None
        list_url = f"{PANEL_URL}/panel/api/inbounds/list"
        try:
            response = self.session.get(list_url, timeout=10)
            if response.ok:
                data = response.json()
                if data.get("success"):
                    self.loaded_inbounds = data.get("obj", [])
                    return self.loaded_inbounds
            return None
        except Exception as e:
            print(f"Error fetching inbounds: {e}")
            return None

    def add_client_to_inbound(self, inbound_id, client_payload):
        """اضافه کردن کلاینت جدید به سرور به صورت بی صدا"""
        if not self.login_to_panel():
            return False, "اتصال به هسته پنل برقرار نشد."

        add_url = f"{PANEL_URL}/panel/api/inbounds/addClient"
        payload = {
            "id": int(inbound_id),
            "settings": json.dumps({
                "clients": [client_payload]
            })
        }
        try:
            response = self.session.post(add_url, json=payload, timeout=10)
            if response.ok:
                res_json = response.json()
                if res_json.get("success"):
                    return True, "موفق"
                return False, res_json.get("msg", "خطای نامشخص")
            return False, "پاسخ ناموفق از API سرور"
        except Exception as e:
            return False, str(e)

    def format_bytes(self, bytes_num):
        """تبدیل بایت به واحدهای ترافیکی"""
        if bytes_num == 0:
            return "0 بایت"
        k = 1024
        sizes = ["بایت", "KB", "MB", "GB", "TB"]
        import math
        i = int(math.floor(math.log(bytes_num) / math.log(k))) if bytes_num > 0 else 0
        p = math.pow(k, i)
        s = round(bytes_num / p, 2)
        return f"{s} {sizes[i]}"

    def build_config_link(self, inbound, client_uuid, email, custom_domain):
        """فرموله کردن و کامپایل کردن لینک اتصال VLESS یا VMess"""
        host_for_config = custom_domain if custom_domain else f"connect.veeed.ir:80"
        protocol = inbound.get("protocol", "vless").lower()

        if protocol == "vless":
            stream_settings_str = inbound.get("streamSettings", "{}")
            try:
                stream_settings = json.loads(stream_settings_str) if isinstance(stream_settings_str, str) else stream_settings_str
            except:
                stream_settings = {}

            stream_type = stream_settings.get("network", "tcp")
            security = stream_settings.get("security", "none")
            path = "%2F"
            host = ""

            if stream_type == "ws":
                ws = stream_settings.get("wsSettings", {})
                path = urllib.parse.quote(ws.get("path", "/"))
                headers = ws.get("headers", {})
                host = headers.get("Host", headers.get("host", ""))
            elif stream_type == "grpc":
                grpc = stream_settings.get("grpcSettings", {})
                path = urllib.parse.quote(grpc.get("serviceName", ""))

            link = f"vless://{client_uuid}@{host_for_config}?encryption=none"
            if host:
                link += f"&host={host}"
            link += f"&path={path}&security={security}&type={stream_type}"
            link += f"#{urllib.parse.quote(email)}"
            return link

        elif protocol == "vmess":
            vmess_config = {
                "v": "2",
                "ps": email,
                "add": host_for_config.split(':')[0],
                "port": inbound.get("port", 80),
                "id": client_uuid,
                "aid": "0",
                "scy": "auto",
                "net": "ws",
                "type": "none",
                "host": "",
                "path": "/",
                "tls": "none",
                "sni": ""
            }
            json_bytes = json.dumps(vmess_config).encode('utf-8')
            base64_config = base64.b64encode(json_bytes).decode('utf-8')
            return f"vmess://{base64_config}"

        return "پروتکل پشتیبانی نمی‌شود"

    def send_telegram_message(self, text):
        """ارسال پیام متنی به چت شما"""
        url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
        payload = {
            "chat_id": ALLOWED_CHAT_ID,
            "text": text,
            "parse_mode": "Markdown"
        }
        try:
            requests.post(url, json=payload, timeout=10)
        except Exception as e:
            print(f"Error sending msg: {e}")

    def process_commands(self, message):
        """پردازش هوشمند دستورات دریافتی از چت خصوصی شما"""
        text = message.get("text", "").strip()
        chat_id = message.get("chat", {}).get("id")

        if chat_id != ALLOWED_CHAT_ID:
            return

        # اگر کاربر در حالت وارد کردن اطلاعات ساخت اکانت تعاملی باشد
        if chat_id in USER_STATES:
            self.handle_creation_states(chat_id, text)
            return

        if text == "/start" or text == "/help":
            help_msg = (
                "🕵️‍♂️ *به ربات مدیریت زنده و سایلنت V3EEED خوش آمدید!*\n\n"
                "دستورات زیر به صورت مستقیم روی API سرور لینوکس ران می‌شوند:\n"
                "📊 `/stats` - وضعیت ترافیک و آمار کل سرور\n"
                "🔌 `/inbounds` - مشاهده لیست اینباندها و پورت‌ها\n"
                "➕ `/create` - ساخت تعاملی اکانت جدید (بی‌صدا)\n"
                "🔍 `/search NAME` - جستجوی حجم و وضعیت یک کاربر خاص\n"
                "ℹ️ `/help` - راهنمای دستورات"
            )
            self.send_telegram_message(help_msg)

        elif text == "/stats":
            self.send_telegram_message("⏳ در حال استخراج و تحلیل آمار زنده سرور...")
            inbounds = self.fetch_inbounds_data()
            if not inbounds:
                self.send_telegram_message("❌ خطا در اتصال محلی به هسته سنایی.")
                return

            total_clients = 0
            active_clients = 0
            expired_clients = 0
            total_traffic = 0

            for inb in inbounds:
                total_traffic += (inb.get("up", 0) + inb.get("down", 0))
                try:
                    settings = json.loads(inb["settings"]) if isinstance(inb["settings"], str) else inb["settings"]
                except:
                    settings = {}
                clients = settings.get("clients", [])
                total_clients += len(clients)

                client_stats = {s["email"]: s for s in inb.get("clientStats", [])}
                for c in clients:
                    email = c.get("email")
                    stat = client_stats.get(email, {})
                    consumed = stat.get("up", 0) + stat.get("down", 0)
                    limit = c.get("totalGB", 0)
                    expiry = c.get("expiryTime", 0)

                    is_expired = expiry > 0 and expiry < (time.time() * 1000)
                    is_quota_exceeded = limit > 0 and consumed >= limit
                    is_active = stat.get("enable", True) and not is_expired and not is_quota_exceeded

                    if is_active:
                        active_clients += 1
                    else:
                        expired_clients += 1

            report = (
                "📊 *گزارش آماری سرور شما (V3EEED)*\n"
                "━━━━━━━━━━━━━━━━━━\n"
                f"👥 *کل کاربران تعریف شده:* `{total_clients}` اکانت\n"
                f"🟢 *اکانت‌های فعال:* `{active_clients}`\n"
                f"🔴 *اکانت‌های منقضی/پایان حجم:* `{expired_clients}`\n"
                f"💾 *ترافیک کل شبکه لینوکس:* `{self.format_bytes(total_traffic)}`"
            )
            self.send_telegram_message(report)

        elif text == "/inbounds":
            inbounds = self.fetch_inbounds_data()
            if not inbounds:
                self.send_telegram_message("❌ خطا در دریافت اطلاعات.")
                return

            response = "🔌 *لیست اینباندهای فعال سرور شما:*\n━━━━━━━━━━━━━━━━━━\n"
            for inb in inbounds:
                try:
                    settings = json.loads(inb["settings"]) if isinstance(inb["settings"], str) else inb["settings"]
                    clients_count = len(settings.get("clients", []))
                except:
                    clients_count = 0
                response += (
                    f"🏷 *نام:* {inb.get('remark')} | ID: `{inb.get('id')}`\n"
                    f"🌐 *پروتکل:* `{inb.get('protocol').upper()}` | پورت: `{inb.get('port')}`\n"
                    f"👥 *کاربران:* `{clients_count}` نفر | ترافیک: `{self.format_bytes(inb.get('up',0)+inb.get('down',0))}`\n"
                    "----------------------------------\n"
                )
            self.send_telegram_message(response)

        elif text.startswith("/search "):
            search_name = text.split(" ", 1)[1].strip()
            inbounds = self.fetch_inbounds_data()
            if not inbounds:
                self.send_telegram_message("❌ خطا در بارگذاری داده‌ها.")
                return

            found = False
            for inb in inbounds:
                try:
                    settings = json.loads(inb["settings"]) if isinstance(inb["settings"], str) else inb["settings"]
                except:
                    settings = {}
                clients = settings.get("clients", [])
                client_stats = {s["email"]: s for s in inb.get("clientStats", [])}

                for c in clients:
                    email = c.get("email", "")
                    if search_name.lower() in email.lower():
                        found = True
                        stat = client_stats.get(email, {})
                        consumed = stat.get("up", 0) + stat.get("down", 0)
                        limit = c.get("totalGB", 0)
                        expiry = c.get("expiryTime", 0)

                        limit_text = self.format_bytes(limit) if limit > 0 else "نامحدود"
                        expiry_text = time.strftime('%Y-%m-%d %H:%M', time.localtime(expiry/1000)) if expiry > 0 else "نامحدود"

                        user_status = "🟢 فعال" if stat.get("enable", True) and (expiry == 0 or expiry > time.time()*1000) and (limit == 0 or consumed < limit) else "🔴 منقضی / غیرفعال"

                        response = (
                            f"👤 *مشخصات کاربر یافت شده:*\n"
                            f"━━━━━━━━━━━━━━━━━━\n"
                            f"📧 *ایمیل:* `{email}`\n"
                            f"🏷 *اینباند لود شده:* {inb.get('remark')}\n"
                            f"⚡ *وضعیت:* {user_status}\n"
                            f"💾 *حجم مصرفی:* `{self.format_bytes(consumed)}` از `{limit_text}`\n"
                            f"📅 *تاریخ انقضاء:* `{expiry_text}`"
                        )
                        self.send_telegram_message(response)
            if not found:
                self.send_telegram_message(f"❌ کاربری با نام `{search_name}` یافت نشد.")

        elif text == "/create":
            inbounds = self.fetch_inbounds_data()
            if not inbounds:
                self.send_telegram_message("❌ خطا در اتصال به هسته پنل سنایی.")
                return

            USER_STATES[chat_id] = {"step": "WAIT_INBOUND"}

            prompt = "➕ *شروع فرآیند ساخت اکانت تعاملی*\n\nلطفاً شناسه (ID) اینباند مقصد را ارسال کنید:\n\n"
            for inb in inbounds:
                prompt += f"🔹 کد اینباند: `{inb.get('id')}` | نام: {inb.get('remark')} ({inb.get('protocol').upper()})\n"

            self.send_telegram_message(prompt)

    def handle_creation_states(self, chat_id, text):
        """مدیریت گام به گام دریافت اطلاعات برای ساخت اکانت بدون خطا"""
        state = USER_STATES[chat_id]

        if text == "/cancel":
            USER_STATES.pop(chat_id, None)
            self.send_telegram_message("❌ عملیات ساخت اکانت لغو شد.")
            return

        if state["step"] == "WAIT_INBOUND":
            try:
                inbound_id = int(text)
                inbounds = self.loaded_inbounds if self.loaded_inbounds else self.fetch_inbounds_data()
                selected = next((i for i in inbounds if i["id"] == inbound_id), None)
                if not selected:
                    self.send_telegram_message("❌ کد وارد شده نامعتبر است. مجدداً ارسال کنید (یا /cancel برای انصراف):")
                    return

                state["inbound_id"] = inbound_id
                state["protocol"] = selected["protocol"]
                state["step"] = "WAIT_EMAIL"
                self.send_telegram_message("👤 نام کاربری یا ایمیل اکانت را وارد کنید (مثال: `parham-shadow`):")
            except:
                self.send_telegram_message("❌ لطفاً فقط عدد شناسه اینباند را وارد کنید:")

        elif state["step"] == "WAIT_EMAIL":
            email = text.strip()
            if len(email) < 3:
                self.send_telegram_message("❌ نام کاربری خیلی کوتاه است. مجدداً ارسال کنید:")
                return

            state["email"] = email
            state["step"] = "WAIT_GB"
            self.send_telegram_message("💾 سقف حجم مصرفی را به *گیگابایت* وارد کنید (عدد خالی بفرستید، مثلاً `20` | یا عدد `0` برای نامحدود):")

        elif state["step"] == "WAIT_GB":
            try:
                gb = float(text)
                state["gb"] = gb
                state["step"] = "WAIT_DAYS"
                self.send_telegram_message("📅 مدت زمان اعتبار اکانت را به *تعداد روز* وارد کنید (مثال: `30` | یا `0` برای دائمی):")
            except:
                self.send_telegram_message("❌ عدد حجم معتبر نیست. لطفاً یک عدد اعشاری یا صحیح ارسال کنید:")

        elif state["step"] == "WAIT_DAYS":
            try:
                days = int(text)
                state["days"] = days
                state["step"] = "WAIT_DOMAIN"
                self.send_telegram_message("🌐 آدرس دامین متصل کلاینت (مثلاً: `connect.veeed.ir:80`) را بنویسید یا برای استفاده از پیش‌فرض عبارت `/skip` را بفرستید:")
            except:
                self.send_telegram_message("❌ تعداد روزها معتبر نیست. لطفاً یک عدد صحیح بفرستید:")

        elif state["step"] == "WAIT_DOMAIN":
            custom_domain = "" if text == "/skip" else text.strip()
            inbound_id = state["inbound_id"]
            email = state["email"]
            gb_val = state["gb"]
            days_val = state["days"]

            total_bytes = int(gb_val * 1073741824) if gb_val > 0 else 0
            expiry_timestamp = int((time.time() + (days_val * 86400)) * 1000) if days_val > 0 else 0
            client_uuid = str(uuid.uuid4())

            client_payload = {
                "id": client_uuid,
                "email": email,
                "limitIp": 0,
                "totalGB": total_bytes,
                "expiryTime": expiry_timestamp,
                "enable": True,
                "tgId": "",
                "subId": ""
            }

            self.send_telegram_message("⏳ در حال ثبت اطلاعات اکانت به صورت سایلنت روی سرور لینوکس...")
            success, msg = self.add_client_to_inbound(inbound_id, client_payload)

            if success:
                inbounds = self.loaded_inbounds if self.loaded_inbounds else self.fetch_inbounds_data()
                selected_inbound = next((i for i in inbounds if i["id"] == inbound_id), None)
                final_link = self.build_config_link(selected_inbound, client_uuid, email, custom_domain)

                limit_text = f"{gb_val} GB" if gb_val > 0 else "نامحدود"
                expiry_text = f"{days_val} روز" if days_val > 0 else "نامحدود (دائمی)"

                success_report = (
                    "👤 *اکانت جدید با موفقیت صادر شد!*\n"
                    "━━━━━━━━━━━━━━━━━━\n"
                    f"📧 *ایمیل کاربر:* `{email}`\n"
                    f"💾 *سقف حجم ترافیک:* `{limit_text}`\n"
                    f"📅 *میزان اعتبار:* `{expiry_text}`\n"
                    f"🔑 *شناسه یکتا (UUID):* \n`{client_uuid}`\n\n"
                    f"🔗 *لینک اتصال نهایی (VLESS/VMess):*\n"
                    f"`{final_link}`\n"
                    "━━━━━━━━━━━━━━━━━━\n"
                    "🤫 _هیچ پیام آلارمی به گروه ادمین‌های تلگرام فرستاده نشد._"
                )
                self.send_telegram_message(success_report)
            else:
                self.send_telegram_message(f"❌ خطا در ساخت اکانت: {msg}")

            USER_STATES.pop(chat_id, None)

    def run_polling(self):
        """لانگ پولینگ تلگرام"""
        print("⚡ V3EEED Silent Accounting Bot Started Successfully.")
        self.send_telegram_message("⚡ *ربات مدیریت بی صدای V3EEED با موفقیت استارت شد!*\nآماده اجرای فرامین شما به صورت زنده هستیم.")

        while True:
            url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/getUpdates"
            params = {"timeout": 30, "offset": self.last_update_id + 1}
            try:
                response = requests.get(url, params=params, timeout=35)
                if response.ok:
                    result = response.json()
                    updates = result.get("result", [])
                    for update in updates:
                        self.last_update_id = update.get("update_id")
                        if "message" in update:
                            self.process_commands(update["message"])
            except Exception as e:
                print(f"Polling error: {e}")
                time.sleep(5)

if __name__ == "__main__":
    bridge = VortexBridge()
    bridge.run_polling()


 🕵️‍♂️ V3EEED - SETUP WIZARD
 به سیستم راه‌اندازی تعاملی ربات V3EEED خوش آمدید.

1. توکن ربات تلگرام خود را وارد کنید:
=> 8668037909:AAGWlvFpik0H6SjTLJjig_YjPPxEdcRqjrs

2. چت آی‌دی شخصی خود را وارد کنید (فقط اعداد):
=> 6304998144
